# Riemer-Quader: Was sagen unsere Würfe über unsere Modelle?

**Ihr gebt die Modelle selbst ein.** Es sind keine Wahrscheinlichkeiten aus einer Vorlage voreingestellt.

1. **Alle Zellen ausführen** (im Menü „Run/Ausführen“). Die beiden Technikzellen dürfen eingeklappt bleiben.
2. Unter **Modelle und Anfangsgewichte** zwei bis sechs Modelle eintragen: Name und sechs Wahrscheinlichkeiten. Jede Zeile muss exakt Summe 1 haben. `0,2`, `0.2`, `1/5` und `20%` sind zulässige Eingaben.
3. Unter **Versuchsaufbau und Start** wählen: reale Würfe erfassen oder simulieren. Für Simulationen muss die Lehrkraft eine eigene, feste Ergebnisverteilung und einen Seed eingeben.
4. Den Versuch starten. Vor dem nächsten Wurf vermuten, welches Ergebnis für welches Modell sprechen würde.

**Wichtig:** Beim realen Quader sind die Ergebniswahrscheinlichkeiten unbekannt. Eine Simulation zieht Zahlen nach einer ausdrücklich festgelegten Verteilung; sie berechnet nicht die Physik eines Quaderwurfs. Wir nehmen für den Vergleich unabhängige Würfe mit gleichbleibenden Wahrscheinlichkeiten an.

Die Oberfläche erscheint nach den beiden eingeklappten Technikzellen. Bei fehlenden Paketen die Anleitung `START.md` verwenden. Bedienelemente funktionieren nur mit laufendem Python-Kernel; eine gespeicherte Ansicht allein ist nicht interaktiv.


In [1]:
"""Exakter Rechenkern der Lernumgebung zum Riemer-Quader. Python >= 3.10."""
from dataclasses import dataclass, replace
from fractions import Fraction
from pathlib import Path
from random import Random
from math import lcm, prod
from datetime import datetime, timezone
import hashlib
import json
import platform
import re

VERSION = "1.0"
Q = Fraction
MAX_WURFE = 1000


def zahl(text):
    """Dezimalpunkt, Dezimalkomma, Bruch oder Prozent; keine Float-Eingabe."""
    if isinstance(text, Q):
        return text
    if type(text) is int:
        return Q(text)
    if not isinstance(text, str):
        raise ValueError("Zahlen als Text, Bruch oder ganze Zahl eingeben.")
    s = text.strip().replace(" ", "").replace(",", ".")
    if not s or len(s) > 40:
        raise ValueError("Leeres oder zu langes Zahlenfeld.")
    try:
        value = Q(s[:-1]) / 100 if s.endswith("%") else Q(s)
    except (ValueError, ZeroDivisionError):
        raise ValueError(f"Ungültige Zahl: {text!r}.") from None
    if value.denominator > 10**6:
        raise ValueError("Für die Lernumgebung höchstens sechs Dezimalstellen verwenden.")
    return value


def verteilung(p, n, bezeichnung):
    if not isinstance(p, tuple) or len(p) != n:
        raise ValueError(f"{bezeichnung}: genau {n} Werte erforderlich.")
    if any(not isinstance(x, Q) or not 0 <= x <= 1 for x in p):
        raise ValueError(f"{bezeichnung}: alle Werte müssen zwischen 0 und 1 liegen.")
    total = sum(p, Q(0))
    if total != 1:
        raise ValueError(f"{bezeichnung}: Summe {total} (≈ {float(total):.8g}), erforderlich ist exakt 1. Keine automatische Normierung.")


@dataclass(frozen=True)
class Modell:
    name: str
    p: tuple[Fraction, ...]

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip() or len(self.name) > 60:
            raise ValueError("Modellnamen mit 1 bis 60 Zeichen eingeben.")
        verteilung(self.p, 6, self.name)


@dataclass(frozen=True)
class Vergleich:
    modelle: tuple[Modell, ...]
    prior: tuple[Fraction, ...]

    def __post_init__(self):
        if not isinstance(self.modelle, tuple) or not 2 <= len(self.modelle) <= 6:
            raise ValueError("Zwei bis sechs Modelle sind möglich.")
        if len({m.name.strip() for m in self.modelle}) != len(self.modelle):
            raise ValueError("Modellnamen müssen verschieden sein.")
        verteilung(self.prior, len(self.modelle), "Anfangsgewichte")


@dataclass(frozen=True)
class Aufbau:
    modus: str
    seed: int = 1
    generator: Modell | None = None
    notiz: str = ""

    def __post_init__(self):
        if self.modus not in ("real", "simulation") or type(self.seed) is not int:
            raise ValueError("Modus und ganzzahliger Seed erforderlich.")
        if self.modus == "simulation" and self.generator is None:
            raise ValueError("Für Simulationen sechs Generatorwahrscheinlichkeiten eingeben.")
        if self.modus == "real" and self.generator is not None:
            raise ValueError("Bei realen Würfen wird kein wahrer Generator vorgegeben.")
        if not isinstance(self.notiz, str) or len(self.notiz) > 5000:
            raise ValueError("Die Versuchsnotiz darf höchstens 5000 Zeichen enthalten.")


@dataclass(frozen=True)
class Revision:
    bei_n: int
    vergleich: Vergleich

    def __post_init__(self):
        if type(self.bei_n) is not int or not 0 <= self.bei_n <= MAX_WURFE:
            raise ValueError("Ungültiger Zeitpunkt einer Modellrevision.")


@dataclass(frozen=True)
class Versuch:
    aufbau: Aufbau
    revisionen: tuple[Revision, ...]
    wurfe: tuple[int, ...] = ()
    aufgedeckt: bool = False

    def __post_init__(self):
        if not isinstance(self.revisionen, tuple) or not self.revisionen or not isinstance(self.wurfe, tuple) or len(self.wurfe) > MAX_WURFE:
            raise ValueError("Fehlender Vergleich oder zu viele Würfe.")
        if type(self.aufgedeckt) is not bool:
            raise ValueError("Ungültiger Aufdeckstatus.")
        if any(type(x) is not int or not 1 <= x <= 6 for x in self.wurfe):
            raise ValueError("Würfe müssen ganze Zahlen zwischen 1 und 6 sein.")

    @property
    def vergleich(self):
        return self.revisionen[-1].vergleich


@dataclass(frozen=True)
class Schritt:
    augenzahl: int
    vorher: tuple[Fraction, ...] | None
    likelihood: tuple[Fraction, ...]
    produkte: tuple[Fraction, ...] | None
    summe: Fraction | None
    nachher: tuple[Fraction, ...] | None


def rechnung(vergleich, wurfe):
    """None bedeutet undefiniert, niemals eine künstliche Gleichverteilung."""
    weights = vergleich.prior
    history, steps = [weights], []
    for x in wurfe:
        if type(x) is not int or not 1 <= x <= 6:
            raise ValueError("Ungültiger Wurf.")
        likelihood = tuple(m.p[x - 1] for m in vergleich.modelle)
        products = tuple(w * p for w, p in zip(weights, likelihood)) if weights is not None else None
        total = sum(products, Q(0)) if products is not None else None
        after = tuple(p / total for p in products) if total else None
        steps.append(Schritt(x, weights, likelihood, products, total, after))
        history.append(after)
        weights = after
    return tuple(history), tuple(steps)


def gemeinsam(vergleich, wurfe):
    counts = tuple(wurfe.count(k) for k in range(1, 7))
    products = tuple(prior * prod((p ** n for p, n in zip(m.p, counts)), start=Q(1))
                     for prior, m in zip(vergleich.prior, vergleich.modelle))
    total = sum(products, Q(0))
    return tuple(x / total for x in products) if total else None


def neu(aufbau, vergleich):
    return Versuch(aufbau, (Revision(0, vergleich),))


def rohfolge(text):
    # Komma ist hier ein Trenner zwischen ganzzahligen Würfen.
    tokens = re.split(r"[\s,;]+", text.strip()) if text.strip() else []
    if not tokens or any(t not in ("1", "2", "3", "4", "5", "6") for t in tokens):
        raise ValueError("Eine Folge wie 3 3 4 5 1 eingeben; nur Augenzahlen 1 bis 6.")
    return tuple(map(int, tokens))


def erfassen(versuch, wurfe):
    if versuch.aufbau.modus != "real":
        raise ValueError("Handeingaben sind nur im Modus reale Würfe möglich.")
    return replace(versuch, wurfe=versuch.wurfe + tuple(wurfe))


def simulationsfolge(aufbau, n):
    if aufbau.modus != "simulation" or not 0 <= n <= MAX_WURFE:
        raise ValueError("Ungültige Simulationsanforderung.")
    rng = Random(aufbau.seed)
    p = aufbau.generator.p
    denominator = lcm(*(x.denominator for x in p))
    bounds, running = [], 0
    for x in p:
        running += x.numerator * (denominator // x.denominator)
        bounds.append(running)
    draws = []
    for _ in range(n):
        ticket = rng.randrange(denominator)
        draws.append(next(i + 1 for i, bound in enumerate(bounds) if ticket < bound))
    return tuple(draws)


def werfen(versuch, n=1):
    if type(n) is not int or n < 1:
        raise ValueError("Positive ganzzahlige Anzahl erforderlich.")
    draws = simulationsfolge(versuch.aufbau, len(versuch.wurfe) + n)
    if draws[:len(versuch.wurfe)] != versuch.wurfe:
        raise ValueError("Gespeicherte Folge passt nicht zu Seed und Zufallszahlengenerator dieser Laufzeit. Zum Nachrechnen bleibt sie nutzbar.")
    return replace(versuch, wurfe=draws)


def entfernen(versuch):
    if not versuch.wurfe:
        return versuch
    return replace(versuch, wurfe=versuch.wurfe[:-1])


def neu_auswerten(versuch, vergleich):
    # Neuberechnung aus dem angegebenen Prior und ALLEN Rohbeobachtungen.
    # Kein erneutes Verwenden des bisherigen Posteriors als Prior.
    return replace(versuch, revisionen=versuch.revisionen + (Revision(len(versuch.wurfe), vergleich),))


def integer_text(n):
    """Auch große exakte Resultate ohne Änderung globaler Python-Sicherheitslimits."""
    if n.bit_length() < 12000:
        return str(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    parts = []
    while n:
        n, rest = divmod(n, 10**9)
        parts.append(rest)
    return sign + str(parts[-1]) + "".join(f"{p:09d}" for p in reversed(parts[:-1]))


def text_integer(s):
    if not isinstance(s, str) or re.fullmatch(r"-?[0-9]+", s) is None:
        raise ValueError("Ungültiger Zähler oder Nenner.")
    if len(s) < 4000:
        return int(s)
    negative = s.startswith("-")
    s = s[1:] if negative else s
    n = 0
    for start in range(0, len(s), 9):
        chunk = s[start:start + 9]
        n = n * 10**len(chunk) + int(chunk)
    return -n if negative else n


def bruch_text(q):
    return integer_text(q.numerator) if q.denominator == 1 else integer_text(q.numerator) + "/" + integer_text(q.denominator)


def bruch_json(q):
    return {"zaehler": integer_text(q.numerator), "nenner": integer_text(q.denominator)}


def aus_bruch_json(d):
    return Q(text_integer(d["zaehler"]), text_integer(d["nenner"]))


def modell_json(m):
    return {"name": m.name, "wahrscheinlichkeiten": [bruch_json(p) for p in m.p]}


def aus_modell_json(d):
    return Modell(d["name"], tuple(aus_bruch_json(p) for p in d["wahrscheinlichkeiten"]))


def vergleich_json(v):
    return {"modelle": [modell_json(m) for m in v.modelle], "prior": [bruch_json(p) for p in v.prior]}


def aus_vergleich_json(d):
    return Vergleich(tuple(aus_modell_json(m) for m in d["modelle"]), tuple(aus_bruch_json(p) for p in d["prior"]))


def versuch_json(v):
    history, _ = rechnung(v.vergleich, v.wurfe)
    return {
        "format": "riemer-quader-schule", "version": 1, "programmversion": VERSION,
        "python": platform.python_version(),
        "rng": "random.Random / randrange; ganzzahlige Intervalle",
        "aufbau": {"modus": v.aufbau.modus, "seed": v.aufbau.seed,
                   "generator": modell_json(v.aufbau.generator) if v.aufbau.generator else None,
                   "notiz": v.aufbau.notiz},
        "revisionen": [{"bei_n": r.bei_n, "vergleich": vergleich_json(r.vergleich)} for r in v.revisionen],
        "wurfe": list(v.wurfe), "haeufigkeiten": [v.wurfe.count(k) for k in range(1, 7)],
        "aufgedeckt": v.aufgedeckt,
        "posterior_verlauf": [[bruch_json(p) for p in row] if row is not None else None for row in history],
        "hinweis": "Modellgewichte gelten nur unter den eingegebenen Modellen und Annahmen. Null bedeutet 0, null/None bedeutet undefiniert.",
    }


def laden_daten(d):
    if d.get("format") != "riemer-quader-schule" or d.get("version") != 1:
        raise ValueError("Dies ist keine unterstützte Versuchsdatei.")
    a = d["aufbau"]
    v = Versuch(Aufbau(a["modus"], a["seed"], aus_modell_json(a["generator"]) if a["generator"] else None, a["notiz"]),
                tuple(Revision(r["bei_n"], aus_vergleich_json(r["vergleich"])) for r in d["revisionen"]),
                tuple(d["wurfe"]), d["aufgedeckt"])
    # Rohdaten sind maßgeblich, nicht die zufällige Folge einer möglicherweise anderen Python-Version.
    computed = versuch_json(v)
    if computed["haeufigkeiten"] != d["haeufigkeiten"] or computed["posterior_verlauf"] != d["posterior_verlauf"]:
        raise ValueError("Datei inkonsistent: Rohdaten und gespeicherte Ergebnisse stimmen nicht überein.")
    return v


def speichern(v, root, name="versuch"):
    """Neuer Ordner pro Export; vorhandene Versuche werden nicht überschrieben."""
    import csv
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", name).strip("_")[:50] or "versuch"
    base = safe + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = root / base
    directory.mkdir()
    payload = json.dumps(versuch_json(v), ensure_ascii=False, indent=2) + "\n"
    (directory / "versuch.json").write_text(payload, encoding="utf-8")
    with (directory / "wurfe.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["wurf", "augenzahl"])
        writer.writerows(enumerate(v.wurfe, 1))
    with (directory / "modellgewichte.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["nach_n_wuerfen", "modell", "exakter_bruch", "dezimal_gerundet", "status"])
        for t, row in enumerate(rechnung(v.vergleich, v.wurfe)[0]):
            for j, m in enumerate(v.vergleich.modelle):
                writer.writerow([t, m.name, bruch_text(row[j]) if row is not None else "", float(row[j]) if row is not None else "", "definiert" if row is not None else "undefiniert"])
    hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in directory.iterdir() if p.is_file()}
    (directory / "sha256.json").write_text(json.dumps(hashes, indent=2) + "\n", encoding="utf-8")
    return directory


In [2]:
"""Bedienoberfläche. Im Notebook wird der Rechenkern vorher ausgeführt."""
import io
from html import escape
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt


def kurz(value):
    if value is None:
        return "nicht definiert"
    if value in (0, 1):
        return str(int(value))
    if value.numerator.bit_length() < 24 and value.denominator.bit_length() < 24:
        return str(value)
    from decimal import Decimal, localcontext
    with localcontext() as c:
        c.prec = 6
        return "≈ " + str(Decimal(value.numerator) / Decimal(value.denominator))


def prozent(value):
    if value is None:
        return "nicht definiert"
    if value in (0, 1):
        return f"{int(value) * 100} % (exakt)"
    from decimal import Decimal, localcontext
    with localcontext() as c:
        c.prec = 5
        if 1 - value < Q(1, 100000):
            gap = Decimal((1 - value).numerator) / Decimal((1 - value).denominator) * 100
            return f"100 % − ca. {gap} %-Punkte"
        number = Decimal(value.numerator) / Decimal(value.denominator) * 100
        return f"≈ {number} %"


def tabelle(headers, rows):
    head = "".join(f"<th>{escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(x))}</td>" for x in row) + "</tr>" for row in rows)
    return f'<div style="overflow-x:auto"><table class="rq-table"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


class QuaderLernumgebung:
    def __init__(self, ausgabeordner="riemer_versuche"):
        self.ausgabeordner = Path(ausgabeordner)
        self.versuch = None
        self.zeilen = []
        self._busy = False
        self.meldung = widgets.HTML()
        self.status = widgets.HTML()
        self.modelltabelle = widgets.HTML()
        self.gewichte = widgets.HTML()
        self.rechenschritt = widgets.HTML()
        self.folge = widgets.HTML()
        self.generator_info = widgets.HTML()
        self.bild = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1100px"))
        self.priorwahl = widgets.ToggleButtons(options=[("Gleiche Anfangsgewichte", "gleich"), ("Eigene Anfangsgewichte", "eigen")])
        self.priorwahl.observe(self.prior_sichtbar, names="value")
        self.modellbox = widgets.VBox()
        self.add_button = widgets.Button(description="Modell hinzufügen", icon="plus", layout=widgets.Layout(width="180px"))
        self.add_button.on_click(lambda b: self.add_model())
        self.mode = widgets.ToggleButtons(options=[("Reale Quaderwürfe eingeben", "real"), ("Würfe simulieren", "simulation")])
        self.seed = widgets.IntText(value=1, description="Seed:", layout=widgets.Layout(width="200px"))
        self.generator_fields = [widgets.Text(placeholder=f"P({k})", layout=widgets.Layout(width="82px")) for k in range(1, 7)]
        self.generator_box = widgets.VBox([
            widgets.HTML("<b>Verteilung für die Simulation – von der Lehrkraft festlegen</b><br>Diese Verteilung erzeugt die Würfe. Sie muss nicht unter den Kandidaten vorkommen. Es wird keine Physik des Quaders simuliert."),
            widgets.HBox(self.generator_fields, layout=widgets.Layout(flex_flow="row wrap")), self.seed])
        self.notiz = widgets.Textarea(placeholder="Zum Beispiel: Gruppe, Quader, Unterlage und Wurfverfahren.", description="Notiz:", layout=widgets.Layout(width="95%", height="70px"))
        self.mode.observe(self.modus_sichtbar, names="value")
        self.start_button = widgets.Button(description="Neuen Versuch starten", button_style="primary", icon="play", layout=widgets.Layout(width="210px"))
        self.recompare_button = widgets.Button(description="Dieselben Würfe neu auswerten", icon="refresh", layout=widgets.Layout(width="260px"))
        self.start_button.on_click(lambda b: self.sicher(self.start))
        self.recompare_button.on_click(lambda b: self.sicher(self.recompare))
        header = widgets.HBox([widgets.HTML("<b>Modellname</b>", layout=widgets.Layout(width="140px"))] +
                               [widgets.HTML(f"<b>P({k})</b>", layout=widgets.Layout(width="78px")) for k in range(1, 7)] +
                               [widgets.HTML("<b>Prior</b>", layout=widgets.Layout(width="82px"))])
        models_form = widgets.VBox([
            widgets.HTML("<b>Eigene Modelle eingeben</b><br>Jede Zeile enthält sechs Wahrscheinlichkeiten mit Summe exakt 1. Beispiele für die Schreibweise: <code>1/6</code>, <code>0,2</code>, <code>20%</code>. Es werden keine Werte vorgegeben und keine Verteilungen automatisch normiert."),
            self.priorwahl, widgets.Box([header, self.modellbox], layout=widgets.Layout(display="flex", flex_flow="column", overflow="auto")),
            self.add_button,
            widgets.HTML("<small>Bei gleichen Anfangsgewichten erhält jedes der J Modelle exakt 1/J. Eigene Anfangsgewichte müssen ebenfalls Summe 1 ergeben.</small>"),
            self.recompare_button,
            widgets.HTML("<small>Neu auswerten: neue Modelle und neuen Prior auf ALLE bisherigen Würfe anwenden. Der Generator und die Beobachtungen bleiben gleich. Der vorige Vergleich wird mitgespeichert.</small>")])
        setup_form = widgets.VBox([self.mode, self.generator_box, self.notiz, self.start_button,
                                   widgets.HTML("<small>Neuer Versuch: Beobachtungsfolge leeren. Vorher bei Bedarf speichern. Aufbauänderungen gelten erst für einen neuen Versuch.</small>")])
        self.forms = widgets.Accordion(children=[models_form, setup_form], selected_index=0)
        self.forms.set_title(0, "1 · Modelle und Anfangsgewichte")
        self.forms.set_title(1, "2 · Versuchsaufbau und Start")

        self.one = widgets.Button(description="1 Wurf", button_style="info", layout=widgets.Layout(width="110px"))
        self.ten = widgets.Button(description="10 weitere", button_style="info", layout=widgets.Layout(width="120px"))
        self.one.on_click(lambda b: self.sicher(lambda: self.draw(1)))
        self.ten.on_click(lambda b: self.sicher(lambda: self.draw(10)))
        self.real_buttons = []
        for k in range(1, 7):
            b = widgets.Button(description=str(k), layout=widgets.Layout(width="45px"))
            b.on_click(lambda b, value=k: self.sicher(lambda: self.record((value,))))
            self.real_buttons.append(b)
        self.sequence = widgets.Text(placeholder="z. B. 3 3 4 5 1", description="Wurffolge:", layout=widgets.Layout(width="330px"))
        self.sequence_button = widgets.Button(description="Folge erfassen", layout=widgets.Layout(width="150px"))
        self.sequence_button.on_click(lambda b: self.sicher(self.record_sequence))
        self.undo = widgets.Button(description="Letzten Wurf entfernen", icon="undo", layout=widgets.Layout(width="205px"))
        self.undo.on_click(lambda b: self.sicher(self.undo_last))
        self.reveal = widgets.Button(description="Simulationsverteilung aufdecken", icon="eye", layout=widgets.Layout(width="275px"))
        self.reveal.on_click(lambda b: self.sicher(self.reveal_generator))
        self.controls_sim = widgets.HBox([self.one, self.ten, self.reveal], layout=widgets.Layout(flex_flow="row wrap"))
        self.controls_real = widgets.VBox([widgets.HBox([widgets.HTML("<b>Geworfene Augenzahl:</b>")] + self.real_buttons),
                                          widgets.HBox([self.sequence, self.sequence_button], layout=widgets.Layout(flex_flow="row wrap"))])
        self.step = widgets.IntSlider(description="Wurf ansehen:", min=1, max=1, value=1, disabled=True,
                                      style={"description_width": "initial"}, layout=widgets.Layout(width="90%"), continuous_update=False)
        self.step.observe(self.select_step, names="value")
        self.filename = widgets.Text(value="quader", description="Dateiname:", layout=widgets.Layout(width="280px"))
        self.save = widgets.Button(description="Versuch speichern", icon="save", layout=widgets.Layout(width="180px"))
        self.save.on_click(lambda b: self.sicher(self.export))
        self.upload = widgets.FileUpload(accept=".json", multiple=False, description="Versuch laden")
        self.upload.observe(self.import_upload, names="value")
        storage = widgets.VBox([widgets.HBox([self.filename, self.save, self.upload], layout=widgets.Layout(flex_flow="row wrap")),
            widgets.HTML("<small>Export: JSON, CSV und SHA-256-Prüfsummen in einem neuen Unterordner. JSON enthält alle Modelle, Priors, Rohwürfe, exakte Ergebnisse und bei Simulationen auch die Generatorverteilung. Zum Laden <code>versuch.json</code> auswählen. Bei Schulservern zusätzlich über die Jupyter-Dateiansicht herunterladen.</small>")])
        self.storage = widgets.Accordion(children=[storage], selected_index=None)
        self.storage.set_title(0, "Speichern und wieder öffnen")
        style = widgets.HTML("""<style>
        .rq-table{border-collapse:collapse;margin:10px 0;font-variant-numeric:tabular-nums;font-size:14px}
        .rq-table th,.rq-table td{padding:8px 12px;border-bottom:1px solid #d5dce3;text-align:right}
        .rq-table th{background:#edf3f8}.rq-table th:first-child,.rq-table td:first-child{text-align:left}
        .rq-note{border-left:4px solid #27608d;padding:10px 14px;background:#f0f6fb;color:#182b3a;margin:8px 0}
        </style>""")
        self.panel = widgets.VBox([style,
            widgets.HTML("<h2>Riemer-Quader · Beobachten und Modelle vergleichen</h2><div class='rq-note'><b>Unter diesen Modellen und Annahmen.</b> Die Gewichte summieren sich zu 1, solange die Aktualisierung definiert ist. Das beweist nicht, dass ein Kandidat den realen Quader angemessen beschreibt.</div>"),
            self.forms, self.meldung, self.status, self.controls_sim, self.controls_real, self.undo,
            self.folge, self.gewichte, self.step, self.rechenschritt, self.bild, self.generator_info,
            self.modelltabelle, self.storage], layout=widgets.Layout(width="100%", max_width="1150px"))
        self.add_model()
        self.add_model()
        self.modus_sichtbar()
        self.render()

    def add_model(self, name=None, p=None, prior=""):
        if len(self.zeilen) >= 6:
            self.meldung.value = "Höchstens sechs Modelle in dieser Lernumgebung."
            return
        fields = [widgets.Text(value=str(x), placeholder=f"P({k + 1})", layout=widgets.Layout(width="78px"))
                  for k, x in enumerate(p if p is not None else ("",) * 6)]
        existing = {r["name"].value for r in self.zeilen}
        default = next(f"Modell {i}" for i in range(1, 100) if f"Modell {i}" not in existing)
        row = {"name": widgets.Text(value=name or default, layout=widgets.Layout(width="140px")), "p": fields,
               "prior": widgets.Text(value=str(prior), placeholder="1/J", layout=widgets.Layout(width="82px"))}
        remove = widgets.Button(description="×", tooltip="Modell entfernen", layout=widgets.Layout(width="34px"))
        remove.on_click(lambda b: self.remove_model(row))
        row["box"] = widgets.HBox([row["name"]] + fields + [row["prior"], remove])
        self.zeilen.append(row)
        self.modellbox.children = tuple(r["box"] for r in self.zeilen)
        self.prior_sichtbar()

    def remove_model(self, row):
        if len(self.zeilen) <= 2:
            self.meldung.value = "Mindestens zwei Modelle für einen Vergleich eingeben."
            return
        self.zeilen.remove(row)
        self.modellbox.children = tuple(r["box"] for r in self.zeilen)
        self.prior_sichtbar()

    def prior_sichtbar(self, change=None):
        for row in self.zeilen:
            row["prior"].disabled = self.priorwahl.value == "gleich"
            if self.priorwahl.value == "gleich":
                row["prior"].value = str(Q(1, len(self.zeilen)))

    def modus_sichtbar(self, change=None):
        self.generator_box.layout.display = "" if self.mode.value == "simulation" else "none"

    def lese_vergleich(self):
        models = []
        for r in self.zeilen:
            name = r["name"].value.strip()
            try:
                models.append(Modell(name, tuple(zahl(f.value) for f in r["p"])))
            except ValueError as e:
                raise ValueError(f"{name or 'Modell ohne Namen'}: {e}") from None
        prior = ((Q(1, len(models)),) * len(models) if self.priorwahl.value == "gleich"
                 else tuple(zahl(r["prior"].value) for r in self.zeilen))
        return Vergleich(tuple(models), prior)

    def sicher(self, action):
        try:
            action()
        except (ValueError, KeyError, TypeError, OSError, ZeroDivisionError) as e:
            self.meldung.value = f'<p style="color:#a52b24"><b>Bitte prüfen:</b> {escape(str(e))}</p>'

    def start(self):
        comparison = self.lese_vergleich()
        generator = Modell("Simulationsverteilung", tuple(zahl(f.value) for f in self.generator_fields)) if self.mode.value == "simulation" else None
        setup = Aufbau(self.mode.value, self.seed.value, generator, self.notiz.value)
        self.versuch = neu(setup, comparison)
        self.forms.selected_index = None
        self.meldung.value = "Neuer Versuch gestartet. Vor jedem Wurf zunächst eine Vermutung formulieren."
        self.render()

    def require(self):
        if self.versuch is None:
            raise ValueError("Zuerst Modelle eingeben und einen Versuch starten.")

    def draw(self, n):
        self.require()
        self.versuch = werfen(self.versuch, n)
        self.meldung.value = "Simulierte Würfe ergänzt. Der festgelegte Generator wurde nicht geändert."
        self.render()

    def record(self, sequence):
        self.require()
        self.versuch = erfassen(self.versuch, sequence)
        self.meldung.value = "Reale Würfe erfasst. Die tatsächlichen Ergebniswahrscheinlichkeiten sind unbekannt."
        self.render()

    def record_sequence(self):
        self.record(rohfolge(self.sequence.value))
        self.sequence.value = ""

    def undo_last(self):
        self.require()
        self.versuch = entfernen(self.versuch)
        self.meldung.value = "Letzten Eintrag entfernt und alle Gewichte aus den übrigen Würfen neu berechnet. Nur Eingabefehler korrigieren; keine unbequemen realen Ergebnisse aussortieren."
        self.render()

    def reveal_generator(self):
        self.require()
        if self.versuch.aufbau.modus != "simulation":
            raise ValueError("Reale Würfe besitzen hier keine bekannte wahre Verteilung.")
        self.versuch = replace(self.versuch, aufgedeckt=True)
        self.meldung.value = "Die zur Simulation festgelegte Verteilung wird unter dem Diagramm angezeigt."
        self.render()

    def recompare(self):
        self.require()
        comparison = self.lese_vergleich()
        self.versuch = neu_auswerten(self.versuch, comparison)
        self.forms.selected_index = None
        self.meldung.value = "Vergleich geändert: dieselben Rohwürfe, neu eingegebene Modelle und Anfangsgewichte. Der bisherige Posterior wurde nicht als Prior wiederverwendet."
        self.render()

    def select_step(self, change=None):
        if not self._busy:
            self.render_step()

    def render_step(self):
        if self.versuch is None or not self.versuch.wurfe:
            self.rechenschritt.value = "<p>Nach dem ersten Wurf erscheint hier die vollständige Aktualisierung.</p>"
            return
        _, steps = rechnung(self.versuch.vergleich, self.versuch.wurfe)
        s = steps[self.step.value - 1]
        if s.vorher is None:
            self.rechenschritt.value = "<p><b>Aktualisierung bereits vorher undefiniert.</b> Die eingegebenen Modelle mit positivem Prior schließen die bisherige Folge aus.</p>"
            return
        rows = [(m.name, kurz(s.vorher[j]), kurz(s.likelihood[j]), kurz(s.produkte[j]),
                 kurz(s.nachher[j]) if s.nachher is not None else "undefiniert") for j, m in enumerate(self.versuch.vergleich.modelle)]
        title = f"<h3>Wurf {self.step.value}: Augenzahl {s.augenzahl}</h3>"
        explanation = "<b>1.</b> Bisheriges Gewicht × Wahrscheinlichkeit der beobachteten Augenzahl.<br><b>2.</b> Jedes Produkt durch die Summe aller Produkte teilen."
        normalizer = f"<p>Summe der Produkte: <b>{escape(kurz(s.summe))}</b>. " + ("Durch 0 darf nicht geteilt werden." if s.summe == 0 else "Durch diese Summe wird geteilt.") + "</p>"
        exactrows = [(m.name, bruch_text(s.vorher[j]), bruch_text(s.likelihood[j]), bruch_text(s.produkte[j]), bruch_text(s.nachher[j]) if s.nachher else "undefiniert") for j, m in enumerate(self.versuch.vergleich.modelle)]
        headers = ["Modell", "Bisheriges Gewicht", f"P({s.augenzahl}) im Modell", "Produkt", "Neues Gewicht"]
        self.rechenschritt.value = title + explanation + tabelle(headers, rows) + normalizer + "<details><summary>Exakte Brüche anzeigen</summary>" + tabelle(headers, exactrows) + "</details>"

    def render(self):
        v = self.versuch
        active = v is not None
        self.recompare_button.disabled = self.save.disabled = not active
        self.undo.disabled = not active or not v.wurfe
        self.controls_sim.layout.display = "" if active and v.aufbau.modus == "simulation" else "none"
        self.controls_real.layout.display = "" if active and v.aufbau.modus == "real" else "none"
        if not active:
            self.status.value = "<p><b>Noch kein Versuch.</b> Zuerst eigene Modelle eingeben, dann den Versuchsaufbau öffnen und starten.</p>"
            self.render_step()
            return
        history, steps = rechnung(v.vergleich, v.wurfe)
        current = history[-1]
        self._busy = True
        self.step.max = max(1, len(v.wurfe))
        self.step.value = max(1, len(v.wurfe))
        self.step.disabled = not v.wurfe
        self._busy = False
        names = ", ".join(m.name for m in v.vergleich.modelle)
        mode = "Reale Würfe" if v.aufbau.modus == "real" else f"Simulation · Seed {v.aufbau.seed}"
        self.status.value = f"<h3>{escape(mode)} · {len(v.wurfe)} Würfe</h3><p><b>Aktiver Vergleich:</b> {escape(names)}. Annahme: unabhängige Würfe mit gleichbleibenden Ergebniswahrscheinlichkeiten.</p>"
        visible = " · ".join(map(str, v.wurfe[-40:])) or "noch keine"
        self.folge.value = f"<p><b>{'Letzte 40' if len(v.wurfe) > 40 else 'Bisherige'} Würfe:</b> {visible}</p>" + tabelle(["Augenzahl"] + list(range(1, 7)), [["Absolute Häufigkeit"] + [v.wurfe.count(k) for k in range(1, 7)]])
        rows = [(m.name, prozent(v.vergleich.prior[j]), prozent(current[j]) if current is not None else "undefiniert") for j, m in enumerate(v.vergleich.modelle)]
        text = "<h3>Aktuelle Gewichte unter diesen Modellen</h3>" + tabelle(["Modell", "Anfangsgewicht", "Nach diesen Würfen"], rows)
        if current is None:
            text += "<div class='rq-note'><b>Kein Posterior definiert.</b> Alle Modelle mit positivem Anfangsgewicht ordnen der beobachteten Folge Wahrscheinlichkeit 0 zu. Die Würfe bleiben gespeichert. Modelle und Eingaben prüfen; es wird kein Ersatzgewicht vergeben.</div>"
        else:
            text += "<p>Summe exakt 1. Anzeigen mit ≈ oder ca. sind gerundet. Ein größtes Gewicht beweist keine gute Beschreibung des realen Quaders.</p>"
        if len(v.revisionen) > 1:
            old = v.revisionen[-2].vergleich
            oldw = rechnung(old, v.wurfe)[0][-1]
            text += "<details open><summary>Voriger Vergleich, auf dieselben aktuellen Würfe angewendet</summary>" + tabelle(["Vorheriges Modell", "Vorheriger Prior", "Gewicht im vorigen Vergleich"], [(m.name, prozent(old.prior[j]), prozent(oldw[j]) if oldw is not None else "undefiniert") for j, m in enumerate(old.modelle)]) + "</details>"
        self.gewichte.value = text
        self.modelltabelle.value = "<details><summary>Aktiv verwendete Modelle und Anfangsgewichte prüfen</summary>" + tabelle(["Modell"] + [f"P({k})" for k in range(1, 7)] + ["Prior"], [(m.name,) + tuple(map(str, m.p)) + (str(v.vergleich.prior[j]),) for j, m in enumerate(v.vergleich.modelle)]) + "</details>"
        self.render_step()
        self.draw_plot(history)
        if v.aufbau.modus == "simulation" and v.aufgedeckt:
            generator = v.aufbau.generator
            included = any(m.p == generator.p for m in v.vergleich.modelle)
            self.generator_info.value = "<div class='rq-note'><b>Aufgedeckte Simulationsverteilung</b><br>" + tabelle([f"P({k})" for k in range(1, 7)], [list(map(str, generator.p))]) + f"Diese Verteilung ist {'unter den aktiven Modellen enthalten' if included else 'in keinem der aktiven Modelle enthalten'}. Das ist Wissen über diese Simulation, kein Befund über einen realen Quader.</div>"
        elif v.aufbau.modus == "simulation":
            self.generator_info.value = "<p>Die Simulationsverteilung ist festgelegt und zunächst ausgeblendet. Das ist eine didaktische Ausblendung, kein Zugriffsschutz.</p>"
        else:
            self.generator_info.value = "<p>Die tatsächliche Verteilung des realen Quaders ist unbekannt. Relative Häufigkeiten sind Beobachtungen, keine exakt bekannten Wahrscheinlichkeiten.</p>"

    def draw_plot(self, history):
        v = self.versuch
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
        for j, m in enumerate(v.vergleich.modelle):
            color = f"C{j}"
            axes[0].plot(range(len(history)), [float(row[j]) if row is not None else float('nan') for row in history], label=m.name, color=color)
            axes[1].plot(range(1, 7), [float(p) for p in m.p], marker=".", alpha=.8, label=m.name, color=color)
        axes[0].set(xlabel="Anzahl Würfe · 0 = Anfangsgewichte", ylabel="Modellgewicht", ylim=(-.02, 1.02), title="Nur innerhalb der gewählten Modellmenge")
        axes[0].legend(fontsize=8)
        if v.wurfe:
            axes[1].scatter(range(1, 7), [v.wurfe.count(k)/len(v.wurfe) for k in range(1, 7)], color="black", marker="x", s=65, label="Relative Häufigkeiten", zorder=5)
        if v.aufbau.modus == "simulation" and v.aufgedeckt:
            axes[1].plot(range(1, 7), [float(p) for p in v.aufbau.generator.p], color="black", linestyle="--", label="Simulationsverteilung")
        axes[1].set(xlabel="Augenzahl", ylabel="Wahrscheinlichkeit / relative Häufigkeit", xticks=range(1, 7), ylim=(-.02, 1.02), title="Vorhersagen und Beobachtungen vergleichen")
        axes[1].legend(fontsize=8)
        for ax in axes:
            ax.grid(alpha=.2)
        fig.tight_layout()
        stream = io.BytesIO()
        fig.savefig(stream, format="png", dpi=120)
        plt.close(fig)
        self.bild.value = stream.getvalue()

    def export(self):
        self.require()
        directory = speichern(self.versuch, self.ausgabeordner, self.filename.value)
        self.meldung.value = "<b>Gespeichert:</b> " + escape(str(directory.resolve()))
        self.last_saved = directory

    def load(self, payload):
        new_run = laden_daten(payload)
        self.versuch = new_run
        self.zeilen.clear()
        self.priorwahl.value = "eigen"
        for m, p in zip(new_run.vergleich.modelle, new_run.vergleich.prior):
            self.add_model(m.name, m.p, str(p))
        self.mode.value = new_run.aufbau.modus
        self.seed.value = new_run.aufbau.seed
        self.notiz.value = new_run.aufbau.notiz
        for i, f in enumerate(self.generator_fields):
            f.value = str(new_run.aufbau.generator.p[i]) if new_run.aufbau.generator else ""
        self.forms.selected_index = None
        self.meldung.value = "Versuch geladen. Alle gespeicherten Ergebnisse wurden aus den Rohwürfen erneut berechnet und geprüft."
        self.render()

    def import_upload(self, change):
        if not self.upload.value:
            return
        def action():
            values = self.upload.value
            item = next(iter(values.values())) if isinstance(values, dict) else values[0]
            content = bytes(item["content"])
            if len(content) > 50_000_000:
                raise ValueError("Die Versuchsdatei ist für diese Lernumgebung zu groß.")
            self.load(json.loads(content.decode("utf-8")))
        self.sicher(action)
        self.upload.value = ()

    def anzeigen(self):
        display(self.panel)
        return self


In [3]:
lernumgebung = QuaderLernumgebung(ausgabeordner="riemer_versuche")
lernumgebung.anzeigen();

## 1 · Ein Wurf – eine begründete Änderung

**Vorher:** Vergleicht die Wahrscheinlichkeiten einer bestimmten Augenzahl in euren Modellen. Welches Modell würde durch dieses Ergebnis relativ zu den anderen gestärkt?

**Nachher:** Erfasst einen realen Wurf oder simuliert genau einen Wurf. Verfolgt die Tabelle: bisheriges Gewicht × Wahrscheinlichkeit des beobachteten Ergebnisses, anschließend Division durch die Summe der Produkte.

- Erklärt den Unterschied zwischen der Spalte „P(Augenzahl) im Modell“ und der Spalte „Neues Gewicht“.
- Ein Modell erhält ein kleineres Gewicht. Ist es damit ausgeschlossen?
- Falls ein Modell vorher Gewicht 0 hatte: Kann es durch die Würfe wieder ein positives Gewicht erhalten?

Am Schieberegler könnt ihr später jeden einzelnen Rechenschritt erneut ansehen. Ein Wert mit `≈` oder `ca.` ist eine gerundete Anzeige. Die Rechnung selbst verwendet exakte Brüche.

## 2 · Längere Folgen und Wiederholungen

Erfasst oder simuliert weitere Würfe. Beschreibt Phasen, in denen ein anderes Modell vorne liegt. Die Grafik beginnt bei **0 Würfen**, also bei den Anfangsgewichten.

Speichert den Versuch. Wiederholt eine Simulation mit identischem Aufbau und Seed: Die gleiche Python-Implementierung erzeugt dieselbe Folge. Verändert dann ausschließlich den Seed. Der Verlauf darf sich ändern.

Bei realen Würfen sollten Unterlage und Wurfverfahren möglichst gleich bleiben. Notiert den Aufbau. Entfernt einen Eintrag nur, um einen Eingabefehler zu korrigieren; sortiert keine Ergebnisse nach ihrer Wirkung auf den Vergleich aus.

## 3 · Was, wenn keines unserer Modelle zutrifft?

Für diese Untersuchung legt die Lehrkraft **vor dem ersten Wurf** eine Simulationsverteilung fest, die nicht unter den eingegebenen Modellen vorkommt. Sie sollte sich von deren Vorhersagen erkennbar unterscheiden. Danach bleibt der Eingabebereich geschlossen.

Lasst die Simulation laufen und beschreibt zunächst nur, was die Modellgewichte aussagen. Deckt anschließend die Simulationsverteilung auf. Prüft, ob sie zu einem eurer Modelle gehört.

**Diskutiert:** Wie kann ein Modell ein sehr großes Gewicht erhalten, obwohl es nicht die verwendete Simulationsverteilung ist? Die Summe der Gewichte ist keine Prüfung, ob die Modellmenge vollständig ist.

Die rechte Grafik vergleicht Modellvorhersagen mit relativen Häufigkeiten. Abweichungen können auch zufällig entstehen. Diese Darstellung allein ist kein statistischer Test und beweist weder die Richtigkeit noch die Falschheit eines Modells.

## 4 · Dieselben Würfe mit anderen Modellen auswerten

Speichert euren Versuch. Öffnet die Modelleingabe, ergänzt einen selbst begründeten Kandidaten und legt die Anfangsgewichte für die **neue** Modellmenge ausdrücklich fest. Drückt **Dieselben Würfe neu auswerten**.

Es werden keine neuen Daten erzeugt. Die Rechnung beginnt mit dem neu angegebenen Prior und verwendet die vollständige vorhandene Wurffolge. Der alte Posterior wird nicht noch einmal als Prior verwendet. Die Ergebnisse des vorigen Vergleichs werden daneben als Tabelle gezeigt und beide Konfigurationen werden gespeichert.

Verändert in einer zweiten Untersuchung nur die Anfangsgewichte. Was bleibt gleich, was verändert sich?

Wenn ihr ein neues Modell erst nach Betrachtung der Daten entwickelt, dokumentiert das. Seine gute Anpassung an diese Daten ist noch keine unabhängige Bestätigung. Prüft Vorhersagen anschließend an neu erhobenen Daten.

**Formuliert euer Ergebnis:** „Unter diesen Modellen und unter diesen Annahmen sprechen die Beobachtungen stärker für … . Daraus folgt noch nicht, dass … .“


## Mathematische Einordnung und Hinweise

<details><summary><b>Bedeutung der Größen und Annahmen</b></summary>

Ein Kandidat $M_j$ legt für die Augenzahlen $k=1,\ldots,6$ die Werte $p_j(k)=P(X=k\mid M_j)$ fest. Bei $J$ Kandidaten ist $\pi_j$ das Anfangsgewicht. Im angenommenen gemeinsamen Wahrscheinlichkeitsmodell beschreibt $\pi$ eine Verteilung über die Kandidaten.

Für die beobachtete Folge $x_1,\ldots,x_n$ und bedingt unabhängige, identisch verteilte Würfe gilt

$$w_j=\frac{\pi_j\prod_{t=1}^{n}p_j(x_t)}{\sum_{\ell=1}^{J}\pi_\ell\prod_{t=1}^{n}p_\ell(x_t)}.$$

Die Gewichte sind Posteriorwahrscheinlichkeiten **innerhalb dieses angenommenen Vergleichsmodells**. Der reale Quader muss durch keinen der Kandidaten exakt beschrieben werden. Deshalb wird die Größe nicht pauschal „Vertrauen“ genannt. Auch das Modell mit dem größten Gewicht kann schlecht passen.

Bei festem Modell bilden die sechs $p_j(k)$ eine Wahrscheinlichkeitsverteilung. Bei einer festgehaltenen Beobachtung $x$ ist $j\mapsto p_j(x)$ die Likelihood des Modells für diesen Wurf. Ihre Werte über den Modellen müssen nicht Summe 1 haben.

Ist der Nenner 0, ist der Posterior nicht definiert. Das Programm behält die Beobachtung und zeigt diesen Zustand an. Ein Modell mit Prior 0 bleibt in diesem endlichen Vergleich ausgeschlossen. Ein positiv gewichtetes Modell bleibt nur dann positiv, wenn es alle beobachteten Ergebnisse mit positiver Wahrscheinlichkeit versieht.

Ein Gleichprior von $1/J$ ist eine ausdrücklich gewählte Setzung. Aus bloßem Nichtwissen folgt er nicht automatisch. Für den Vergleich mit veränderten Priors werden immer dieselben Rohdaten verwendet.

</details>

<details><summary><b>Unterrichtlicher Aufbau und Reichweite</b></summary>

Ein möglicher Ablauf: Modelle entwickeln und auf Summe 1 prüfen; einzelne Aktualisierungen begründen; längere Wurffolgen auswerten; fehlende Kandidaten untersuchen; Modellmenge und Priors variieren. Die ersten Schritte können mit zwei Modellen beginnen. Weitere Kandidaten kommen später hinzu.

Für den Einstieg sind positive Wahrscheinlichkeiten günstig. Wahrscheinlichkeiten 0 und Anfangsgewichte 0 lassen sich anschließend gezielt untersuchen. Erklärungen zum tatsächlichen Riemer-Quader müssen Wurfverfahren und Material berücksichtigen; dieses Notebook leitet keine Wahrscheinlichkeiten aus Kantenlängen oder Flächeninhalten her.

Die Modellmenge ist frei eingegeben und nicht als vollständig behauptet. Es existieren unendlich viele mögliche Wahrscheinlichkeitsvektoren für die sechs Ergebnisse. Ein endlicher Modellvergleich entscheidet nicht von selbst, welche weiteren Modelle untersucht werden sollten.

Die im Aufbau eingegebene Generatorverteilung bleibt während eines Laufs fest. Das Programm wählt nicht pro Wurf einen Kandidaten aus. Im Modus „real“ gibt es keinen bekannten Generator zum Aufdecken. Die Eingabefelder und Exportdateien einer Simulation enthalten die Generatorwerte; das Ausblenden ist nur didaktisch, kein Zugriffsschutz.

</details>

<details><summary><b>Speicherung, Reproduzierbarkeit und technische Grenzen</b></summary>

Der Export erzeugt bei jedem Speichern einen neuen Unterordner unter `riemer_versuche`. Er enthält `versuch.json`, `wurfe.csv`, `modellgewichte.csv` und SHA-256-Prüfsummen. JSON speichert Zähler und Nenner als Strings, außerdem Aufbau, Seed, Kandidaten, Prior, Modellrevisionen, Notiz, Rohfolge und alle aktuellen Posteriorstände. Nullgewichte werden von nicht definierten Posteriors unterschieden. Beim Laden wird der gesamte Verlauf erneut berechnet und mit der Datei verglichen.

Die rationalen Simulationseingaben werden durch gleichverteilte ganzzahlige Ziehungen umgesetzt; keine Float-Rundung verändert dabei die gewählten Intervalle. Es handelt sich um einen deterministischen Pseudozufallszahlengenerator. Die gespeicherte Beobachtungsfolge ist für langfristige Nachrechnungen maßgeblich, da ein Seed allein nicht jede künftige Implementierung festlegt. Bei einer inkompatiblen RNG-Folge ist Nachrechnen möglich; das Fortsetzen der Simulation wird mit einer Meldung verhindert.

Für einen neuen Lauf wird die alte Anzeige ersetzt. Vorher bei Bedarf speichern. Modellrevisionen dokumentieren die eingegebenen Vergleiche; das Programm führt kein lückenloses Protokoll aller Tastatureingaben oder gelöschter fehlerhafter Würfe.

Die Lernumgebung erlaubt zwei bis sechs Kandidaten und höchstens 1000 Würfe pro Lauf. Sehr große Zähler und Nenner können die exakte Rechnung und die Anzeige verlangsamen. Grafiken sind gerundet. Der Kern arbeitet ohne NumPy oder pandas; für die Oberfläche werden Matplotlib und ipywidgets benötigt.

</details>
